In [ ]:
!pip -q install -U transformers datasets accelerate evaluate scikit-learn

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments
from transformers import Trainer
from datasets import load_dataset, concatenate_datasets


In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)  # Binary classification

In [ ]:
dataset = load_dataset("imdb", split="train")  # Load full train set
print("Loading..")

In [ ]:

# Ensure equal number of positive and negative samples
positive_samples = dataset.filter(lambda x: x["label"] == 1).shuffle(seed=42).select(range(1000))
negative_samples = dataset.filter(lambda x: x["label"] == 0).shuffle(seed=42).select(range(1000))

# Merge and shuffle dataset properly
balanced_dataset = concatenate_datasets([positive_samples, negative_samples]).shuffle(seed=42)

def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors="pt"  # Ensure tensors are properly formatted
    )

tokenized_dataset = balanced_dataset.map(preprocess_function, batched=True, remove_columns=["text"])

In [ ]:
print("Sample data after tokenization:")
print(tokenized_dataset[:5])


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",  # Evaluate after each epoch
    save_strategy="epoch",
    learning_rate=5e-5,  # Increase learning rate slightly
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,  # Increase training epochs
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    lr_scheduler_type="linear",  # Helps adjust learning rate dynamically
)

print("Done")

In [ ]:
# Move Model to GPU Before Training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)  # Move model to GPU for faster training

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset.select(range(1500)),  # Train on 1,500 samples
    eval_dataset=tokenized_dataset.select(range(1500, 2000)),  # Eval on 500 samples
)

trainer.train()

In [ ]:
model.save_pretrained("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")

In [ ]:

def classify_text(text):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Detect GPU or fallback to CPU
    model.to(device)  # Move model to the detected device

    inputs = tokenizer(text, return_tensors="pt", max_length=256, truncation=True)
    inputs = {key: value.to(device) for key, value in inputs.items()}  # Move input tensors to the same device

    outputs = model(**inputs)
    prediction = torch.argmax(outputs.logits).item()
    return "Positive" if prediction == 1 else "Negative"

In [ ]:
sample_text = "This movie was amazing! I loved every moment of it."
print("Sentiment:", classify_text(sample_text))

In [ ]:
print("Sentiment:", classify_text("I absolutely loved this film! It was fantastic."))
print("Sentiment:", classify_text("An incredible performance by the cast. A must-watch!"))
print("Sentiment:", classify_text("The storyline was beautiful and touching."))
print("Sentiment:", classify_text("This is one of the best movies I’ve ever seen."))
print("Sentiment:", classify_text("A masterpiece! Brilliant acting and a compelling plot."))

# Test Negative Sentences
print("Sentiment:", classify_text("This was the worst movie I have ever watched."))
print("Sentiment:", classify_text("Terrible acting, bad plot, and a complete waste of time."))
print("Sentiment:", classify_text("I was bored to death, nothing exciting ever happened."))
print("Sentiment:", classify_text("I regret watching this movie. It was awful."))
print("Sentiment:", classify_text("Disappointing in every way. Don't waste your time."))
